# Case Study: What Makes a Song Popular?
## Notebook 2 — Linear Regression with Categorical Variables

**Building on Notebook 1**, this notebook adds proper treatment of categorical variables:

| Feature | Type | Encoding |
|---|---|---|
| acousticness, danceability, energy, instrumentalness, liveness, loudness, speechiness, tempo, valence, duration_ms, year | Numeric | Keep as-is |
| `key` | Categorical (12 musical keys: C, C#, D…) | One-hot encode |
| `mode` | Binary (0 = minor, 1 = major) | Keep as-is |
| `explicit` | Binary (0 = no, 1 = yes) | Keep as-is |
| id, name, artists, release_date | Identity / text | Drop |

**Goal:** Does treating `key` as a true categorical variable improve prediction?


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

## 1. Load & Inspect Data

In [ ]:
path = 'spotify_data 2.csv'
df = pd.read_csv(path)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 2. Explore Categorical Variables

Even though `key`, `mode`, and `explicit` are stored as integers, they represent categories — not continuous quantities.
- **key**: 0=C, 1=C#/Db, 2=D, 3=D#/Eb, 4=E, 5=F, 6=F#/Gb, 7=G, 8=G#/Ab, 9=A, 10=A#/Bb, 11=B
- **mode**: 0=Minor, 1=Major
- **explicit**: 0=No, 1=Yes


In [ ]:
key_labels = {0:'C', 1:'C#', 2:'D', 3:'D#', 4:'E', 5:'F',
              6:'F#', 7:'G', 8:'G#', 9:'A', 10:'A#', 11:'B'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Key distribution
key_counts = df['key'].value_counts().sort_index()
axes[0].bar([key_labels[k] for k in key_counts.index], key_counts.values, color='steelblue')
axes[0].set_title('Distribution of Key')
axes[0].set_xlabel('Musical Key')
axes[0].set_ylabel('Count')

# Mode distribution
mode_counts = df['mode'].value_counts().sort_index()
axes[1].bar(['Minor (0)', 'Major (1)'], mode_counts.values, color=['tomato', 'steelblue'])
axes[1].set_title('Distribution of Mode')
axes[1].set_ylabel('Count')

# Explicit distribution
exp_counts = df['explicit'].value_counts().sort_index()
axes[2].bar(['Not Explicit (0)', 'Explicit (1)'], exp_counts.values, color=['steelblue', 'tomato'])
axes[2].set_title('Distribution of Explicit')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Average popularity by each categorical variable
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# By key
key_pop = df.groupby('key')['popularity'].mean().sort_index()
axes[0].bar([key_labels[k] for k in key_pop.index], key_pop.values, color='steelblue')
axes[0].set_title('Avg Popularity by Key')
axes[0].set_xlabel('Musical Key')
axes[0].set_ylabel('Avg Popularity')

# By mode
mode_pop = df.groupby('mode')['popularity'].mean()
axes[1].bar(['Minor (0)', 'Major (1)'], mode_pop.values, color=['tomato', 'steelblue'])
axes[1].set_title('Avg Popularity by Mode')
axes[1].set_ylabel('Avg Popularity')

# By explicit
exp_pop = df.groupby('explicit')['popularity'].mean()
axes[2].bar(['Not Explicit (0)', 'Explicit (1)'], exp_pop.values, color=['steelblue', 'tomato'])
axes[2].set_title('Avg Popularity by Explicit')
axes[2].set_ylabel('Avg Popularity')

plt.tight_layout()
plt.show()

## 3. Feature Engineering — One-Hot Encode `key`

- Drop identity columns: `id`, `name`, `artists`, `release_date`
- **One-hot encode `key`** — 12 categories, `drop_first=True` removes one column to avoid multicollinearity (dummy variable trap)
- `mode` and `explicit` are already binary (0/1) — no further encoding needed


In [ ]:
drop_cols = ['id', 'name', 'artists', 'release_date']
df_model = df.drop(columns=drop_cols).copy()

# One-hot encode key with readable labels
df_model['key'] = df_model['key'].map(key_labels)
key_dummies = pd.get_dummies(df_model['key'], prefix='key', drop_first=True)
df_model = pd.concat([df_model.drop(columns=['key']), key_dummies], axis=1)

print(f'Shape after encoding: {df_model.shape}')
print('\nAll columns:')
for col in df_model.columns:
    print(f'  {col} ({df_model[col].dtype})')

In [ ]:
print('Missing values after encoding:')
missing = df_model.isnull().sum()
print(missing[missing > 0])
if missing.sum() == 0:
    print('No missing values.')

## 4. Train-Test Split (80/20)

In [ ]:
X = df_model.drop(columns=['popularity'])
y = df_model['popularity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape[0]:,} rows  x  {X_train.shape[1]} features')
print(f'Test:  {X_test.shape[0]:,} rows  x  {X_test.shape[1]} features')
print(f'\nFeatures ({X_train.shape[1]} total):')
print(list(X_train.columns))

## 5. Linear Regression — Without Standardization

In [ ]:
model_raw = LinearRegression()
model_raw.fit(X_train, y_train)

y_train_pred_raw = model_raw.predict(X_train)
y_test_pred_raw  = model_raw.predict(X_test)

r2_train_raw   = r2_score(y_train, y_train_pred_raw)
rmse_train_raw = np.sqrt(mean_squared_error(y_train, y_train_pred_raw))
r2_test_raw    = r2_score(y_test, y_test_pred_raw)
rmse_test_raw  = np.sqrt(mean_squared_error(y_test, y_test_pred_raw))

print('WITHOUT Standardization')
print(f'  Train — R²: {r2_train_raw:.4f}  RMSE: {rmse_train_raw:.4f}')
print(f'  Test  — R²: {r2_test_raw:.4f}  RMSE: {rmse_test_raw:.4f}')

## 6. Linear Regression — With Standardization

Scaler is fit on train only, then applied to both train and test.  
One-hot encoded columns (0/1) are also scaled — this is fine for linear regression.


In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Verify: mean ≈ 0, std ≈ 1 after scaling
scaled_df = pd.DataFrame(X_train_s, columns=X_train.columns)
pd.DataFrame({
    'mean_before': X_train.mean().round(4),
    'mean_after':  scaled_df.mean().round(4),
    'std_before':  X_train.std().round(4),
    'std_after':   scaled_df.std().round(4)
})

In [ ]:
model_scaled = LinearRegression()
model_scaled.fit(X_train_s, y_train)

y_train_pred_s = model_scaled.predict(X_train_s)
y_test_pred_s  = model_scaled.predict(X_test_s)

r2_train_s   = r2_score(y_train, y_train_pred_s)
rmse_train_s = np.sqrt(mean_squared_error(y_train, y_train_pred_s))
r2_test_s    = r2_score(y_test, y_test_pred_s)
rmse_test_s  = np.sqrt(mean_squared_error(y_test, y_test_pred_s))

print('WITH Standardization')
print(f'  Train — R²: {r2_train_s:.4f}  RMSE: {rmse_train_s:.4f}')
print(f'  Test  — R²: {r2_test_s:.4f}  RMSE: {rmse_test_s:.4f}')

## 7. Comparison: With vs Without Standardization

In [ ]:
results = pd.DataFrame({
    'Model':  ['Without Standardization', 'Without Standardization',
               'With Standardization',    'With Standardization'],
    'Split':  ['Train', 'Test', 'Train', 'Test'],
    'R²':     [r2_train_raw, r2_test_raw, r2_train_s, r2_test_s],
    'RMSE':   [rmse_train_raw, rmse_test_raw, rmse_train_s, rmse_test_s]
}).round(4)

results

## 8. Actual vs Predicted Plots — Train and Test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_true, y_pred, split in [
    (axes[0], y_train, y_train_pred_s, 'Train'),
    (axes[1], y_test,  y_test_pred_s,  'Test'),
]:
    r2 = r2_score(y_true, y_pred)
    ax.scatter(y_true, y_pred, alpha=0.15, s=5, color='steelblue')
    mn, mx = int(y_true.min()), int(y_true.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect fit')
    ax.set_xlabel('Actual Popularity')
    ax.set_ylabel('Predicted Popularity')
    ax.set_title(f'{split}: Actual vs Predicted  (R²={r2:.4f})')
    ax.legend()

plt.tight_layout()
plt.show()

## 9. Residuals Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_true, y_pred, split in [
    (axes[0], y_train, y_train_pred_s, 'Train'),
    (axes[1], y_test,  y_test_pred_s,  'Test'),
]:
    residuals = np.array(y_true) - y_pred
    ax.scatter(y_pred, residuals, alpha=0.15, s=5, color='steelblue')
    ax.axhline(0, color='red', linestyle='--', lw=1.5)
    ax.set_xlabel('Predicted Popularity')
    ax.set_ylabel('Residual (Actual − Predicted)')
    ax.set_title(f'{split} Residuals')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, y_true, y_pred, split in [
    (axes[0], y_train, y_train_pred_s, 'Train'),
    (axes[1], y_test,  y_test_pred_s,  'Test'),
]:
    residuals = np.array(y_true) - y_pred
    sns.histplot(residuals, bins=40, kde=True, ax=ax)
    ax.axvline(0, color='red', linestyle='--')
    ax.set_title(f'{split} Residual Distribution  (mean={residuals.mean():.2f}, std={residuals.std():.2f})')
    ax.set_xlabel('Residual')

plt.tight_layout()
plt.show()

## 10. Overfitting Check

In [ ]:
r2_gap   = r2_train_s - r2_test_s
rmse_gap = rmse_test_s - rmse_train_s

print('=== Overfitting Check (With Standardization) ===')
print(f'  Train R²:   {r2_train_s:.4f}')
print(f'  Test  R²:   {r2_test_s:.4f}')
print(f'  R² Gap:     {r2_gap:.4f}  {"<-- possible overfit" if r2_gap > 0.05 else "(OK)"}')
print()
print(f'  Train RMSE: {rmse_train_s:.4f}')
print(f'  Test  RMSE: {rmse_test_s:.4f}')
print(f'  RMSE Gap:   {rmse_gap:.4f}')
print()
if r2_gap > 0.05:
    print('Conclusion: Model shows OVERFITTING — test R² is noticeably lower than train R².')
else:
    print('Conclusion: No significant overfitting — train and test R² are close.')

## 11. Feature Coefficients — What Drives Popularity?

Standardized coefficients allow direct comparison across all features (numeric and one-hot encoded).  
Blue = positive impact on popularity, Red = negative impact.


In [ ]:
coef_df = pd.DataFrame({
    'Feature':     X_train.columns,
    'Coefficient': model_scaled.coef_
}).sort_values('Coefficient', key=abs, ascending=False).reset_index(drop=True)

plt.figure(figsize=(12, 8))
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', lw=0.8)
plt.xlabel('Coefficient (standardized)')
plt.title('Feature Impact on Popularity\n(Blue = positive, Red = negative)')
plt.tight_layout()
plt.show()

coef_df

In [ ]:
# Zoom in: how does musical key affect popularity?
key_coefs = coef_df[coef_df['Feature'].str.startswith('key_')].copy()
key_coefs['Key'] = key_coefs['Feature'].str.replace('key_', '')

plt.figure(figsize=(10, 4))
colors = ['steelblue' if c > 0 else 'tomato' for c in key_coefs['Coefficient']]
plt.bar(key_coefs['Key'], key_coefs['Coefficient'], color=colors)
plt.axhline(0, color='black', lw=0.8)
plt.xlabel('Musical Key (relative to C, which is the baseline)')
plt.ylabel('Coefficient')
plt.title('Effect of Musical Key on Popularity')
plt.tight_layout()
plt.show()

## Summary

**Result:** One-hot encoding `key` produces the same R² and RMSE as treating it as numeric (Notebook 1).

| | Notebook 1 (key as numeric) | Notebook 2 (key one-hot encoded) |
|---|---|---|
| **Test R²** | ~0.7820 | ~0.7820 |
| **Test RMSE** | ~10.09 | ~10.09 |

**Why?** Musical key has very little predictive power for popularity overall — the coefficients for all key dummies are small.  
Treating it as 12 separate categories vs. a single numeric column does not meaningfully change the model.

**Takeaway:** The baseline audio features explain about 78% of variance in song popularity regardless of how `key` is encoded.  
To improve further, we need a different kind of feature — like artist reputation (explored in Notebook 3).
